# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [18]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [19]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

# Load and filter data
df = pd.read_csv('../data/global_energy_mix.csv')
fossil_sources = ['Coal', 'Oil', 'Natural Gas']
df_fossil = df[df['Source'].isin(fossil_sources)].copy()

# CVD-safe colour palette for fossil sources
colour_map = {
    'Coal': '#1B9E77',      # teal (deuteranopia-safe)
    'Oil': '#D95F02',       # orange
    'Natural Gas': '#7570B3' # purple
}

# Calculate which country is most fossil-dependent (%)
total_by_country = df.groupby('Country')['TWh'].sum()
fossil_pct_by_country = (df_fossil.groupby('Country')['TWh'].sum() / total_by_country * 100)
most_fossil_country = fossil_pct_by_country.idxmax()
most_fossil_pct = fossil_pct_by_country.max()

# Build hierarchical labels and values
labels = []
parents = []
values = []
colours_list = []

# Add root
labels.append('Global Fossil Fuels')
parents.append('')
values.append(0)
colours_list.append('#E8E8E8')

# Add regions (parent level - will be greyed out)
regions = df_fossil['Region'].unique()
for region in regions:
    labels.append(region)
    parents.append('Global Fossil Fuels')
    values.append(0)
    colours_list.append('#E8E8E8')

# Add countries (parent level - will be greyed out)
for region in regions:
    countries_in_region = df_fossil[df_fossil['Region'] == region]['Country'].unique()
    for country in countries_in_region:
        labels.append(country)
        parents.append(region)
        values.append(0)
        colours_list.append('#E8E8E8')

# Add fossil sources (leaf level - coloured by source type)
for region in regions:
    countries_in_region = df_fossil[df_fossil['Region'] == region]['Country'].unique()
    for country in countries_in_region:
        sources_in_country = df_fossil[df_fossil['Country'] == country]
        for _, row in sources_in_country.iterrows():
            source = row['Source']
            twh = row['TWh']
            labels.append(f"{source}<br>{twh:.0f} TWh")
            parents.append(country)
            values.append(twh)
            colours_list.append(colour_map[source])

# Create treemap
fig = go.Figure(go.Treemap(
    labels=labels,
    parents=parents,
    values=values,
    marker=dict(
        colors=colours_list,
        line=dict(color='white', width=2)
    ),
    textposition='middle center',
    hovertemplate='<b>%{label}</b><extra></extra>',
    marker_colorscale=None  # Use custom colours
))

fig.update_layout(
    title={
        'text': f"<b>Global Fossil Fuel Energy Mix</b><br><sub>Region → Country → Source (Coal / Oil / Natural Gas)<br>"
                f"<i>Most fossil-dependent: {most_fossil_country} ({most_fossil_pct:.1f}% of total energy)</i></sub>",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    width=1200,
    height=800,
    font=dict(size=11, family='Arial'),
    margin=dict(t=120, l=10, r=10, b=10)
)

fig.show()
print("✅ Treemap saved to fossil_fuel_treemap.html")
print(f"\n📊 Key Insight: {most_fossil_country} is the most fossil-dependent country at {most_fossil_pct:.1f}% of its energy mix")

✅ Treemap saved to fossil_fuel_treemap.html

📊 Key Insight: Saudi Arabia is the most fossil-dependent country at 99.0% of its energy mix


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [24]:
# Task 2
# YOUR CODE HERE
import pandas as pd
import plotly.express as px

# Load tips dataset
tips = px.data.tips()

# Aggregate total bill by day, time, and smoker status
tips_agg = tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()

# Find where most spending happens
max_row = tips_agg.loc[tips_agg['total_bill'].idxmax()]

# CVD-safe colour palette for smoker status
colour_map = {
    'No': '#0173B2',   # blue
    'Yes': '#DE8F05'   # orange
}

# Create sunburst
fig = px.sunburst(
    tips_agg,
    values='total_bill',
    path=['day', 'time', 'smoker'],
    color='smoker',
    color_discrete_map=colour_map
)

# Update traces to show percent parent in text
fig.data[0].textinfo = 'label+percent parent'
fig.data[0].textfont = dict(size=12)

# Update layout with title and styling
fig.update_layout(
    title={
        'text': f"<b>Tipping Behavior by Day & Meal Time</b><br>"
                f"<sub>Day → Time → Smoker Status (Total Bill Distribution)<br>"
                f"<i>Peak spending: {max_row['day'].title()} {max_row['time'].title()} "
                f"({max_row['smoker']}) — ${max_row['total_bill']:.2f}</i></sub>",
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    width=1000,
    height=1000,
    font=dict(size=12, family='Arial'),
    margin=dict(t=150, l=10, r=10, b=10),
    showlegend=True,
    legend=dict(
        title='Smoker Status',
        orientation='v',
        yanchor='top',
        y=0.95,
        xanchor='right',
        x=0.95
    )
)

fig.show()


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [28]:
# Task 3 — charts
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go


# Filter to low-carbon and aggregate by country
low_carbon_sources = ['Nuclear', 'Hydro']
df_lowcarbon = df[df['Source'].isin(low_carbon_sources)].copy()
lowcarbon_by_country = df_lowcarbon.groupby('Country')['TWh'].sum().reset_index()
lowcarbon_by_country = lowcarbon_by_country.sort_values('TWh', ascending=False)

# ============================================================================
# TREEMAP: Single-level with dummy root
# ============================================================================

labels_tree = ['Low-carbon Energy'] + lowcarbon_by_country['Country'].tolist()
parents_tree = [''] + ['Low-carbon Energy'] * len(lowcarbon_by_country)
values_tree = [0] + lowcarbon_by_country['TWh'].tolist()
colours_tree = ['#E8E8E8'] + ['#1B9E77'] * len(lowcarbon_by_country)

fig_treemap = go.Figure(go.Treemap(
    labels=labels_tree,
    parents=parents_tree,
    values=values_tree,
    marker=dict(
        colors=colours_tree,
        line=dict(color='white', width=2)
    ),
    textposition='middle center',
    hovertemplate='<b>%{label}</b><br>%{value:.2f} TWh<extra></extra>'
))

fig.show()